In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

df = pd.read_csv('../data/berek_update.csv')
print(f'Shape: {df.shape}')

Shape: (177421, 25)


### Backward Search

As a benchmark, we run a backward stepwise search using BIC as the selection criterion, starting from the full model and removing variables one at a time as long as BIC improves. The ownership variables (`foreign_majority`, `state_majority`) are forced into the model throughout. We use backward rather than forward selection because we do not expect the true model to be especially sparse, and domain knowledge suggests that most covariates are plausibly informative and should therefore be included in the initial specification.

In [5]:
# Backward search with BIC
# Start with full model (all variables, no interactions)

all_vars = [
    'foreign_majority', 'state_majority', 'union_presence',
    'experience', 'experience_sq', 'tenure_years', 'total_hours',
    'sex', 'education_5cat', 'position_5cat', 'contract_type',
    'fulltime', 'any_supplement', 'firm_size_4cat', 'collective_agreement',
    'industry_13cat', 'settlement_type', 'nuts2_region'
]

# Forced variables — never removed
forced_vars = ['foreign_majority']

selected = all_vars.copy()

full_model = smf.ols(f"log_earnings ~ {' + '.join(selected)}", data=df).fit()
print(f'Full model BIC: {full_model.bic:.0f}')
print('-' * 60)

while True:
    current_bic = smf.ols(f"log_earnings ~ {' + '.join(selected)}", data=df).fit().bic
    best_bic = current_bic
    worst_var = None
    
    for var in selected:
        if var in forced_vars:
            continue
        candidate = [v for v in selected if v != var]
        model = smf.ols(f"log_earnings ~ {' + '.join(candidate)}", data=df).fit()
        if model.bic < best_bic:
            best_bic = model.bic
            worst_var = var

    if worst_var is None:
        break
    
    selected.remove(worst_var)
    print(f'Removed: {worst_var:<30} BIC: {best_bic:.0f}')

print('-' * 60)
print(f'Final variables: {selected}')

Full model BIC: 167648
------------------------------------------------------------
Removed: union_presence                 BIC: 167637
------------------------------------------------------------
Final variables: ['foreign_majority', 'state_majority', 'experience', 'experience_sq', 'tenure_years', 'total_hours', 'sex', 'education_5cat', 'position_5cat', 'contract_type', 'fulltime', 'any_supplement', 'firm_size_4cat', 'collective_agreement', 'industry_13cat', 'settlement_type', 'nuts2_region']


The search removes only `union_presence` (BIC: 167,637 vs 167,648), arriving at the same specification as our theory-driven baseline model. This provides independent confirmation that our variable selection was sound, the data-driven approach agrees with the economically motivated spec.